In [0]:
dbutils.widgets.text("catalog", "claims_dev")

In [0]:
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog")
raw = f"/Volumes/{catalog}/landing/raw"
chk = f"/Volumes/{catalog}/landing/checkpoints"


def ingest(source, fmt, table):
    stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", fmt)
        .option("cloudFiles.schemaLocation", f"{chk}/{table}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(f"{raw}/{source}")
        .withColumn("_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    query = (
        stream.writeStream
        .option("checkpointLocation", f"{chk}/{table}/state")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{catalog}.bronze.{table}")
    )
    query.awaitTermination()
    return spark.table(f"{catalog}.bronze.{table}").count()

In [0]:
{
    "claim_events_app": ingest("claim_events/app", "json", "claim_events_app"),
    "claim_events_tpa": ingest("claim_events/tpa", "avro", "claim_events_tpa"),
    "policies_cdc": ingest("policies_cdc", "json", "policies_cdc"),
    "investigations": ingest("investigations", "parquet", "investigations"),
}

In [0]:
display(spark.sql(f"""
    SELECT 'app' AS source, count(*) AS rows, count(DISTINCT arrival_date) AS days,
           count(_rescued_data) AS rescued, count(channel) AS with_channel
    FROM {catalog}.bronze.claim_events_app
    UNION ALL
    SELECT 'tpa', count(*), count(DISTINCT arrival_date), count(_rescued_data), count(channel)
    FROM {catalog}.bronze.claim_events_tpa
"""))

In [0]:
for table, fmt in [("workshops", "ORC"), ("coverages", "PARQUET")]:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.{table}")
    result = spark.sql(f"""
        COPY INTO {catalog}.bronze.{table}
        FROM (SELECT *, _metadata.file_path AS _file, current_timestamp() AS _ingested_at FROM '{raw}/{table}')
        FILEFORMAT = {fmt}
        COPY_OPTIONS ('mergeSchema' = 'true')
    """).collect()[0]
    print(table, result["num_inserted_rows"])